In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# 1. Load Synthetic Data
# Note: The dataset is synthetic and intentionally small to demonstrate reasoning behavior.
concepts_df = pd.read_csv("data/concepts.csv")
prerequisites_df = pd.read_csv("data/prerequisites.csv")
resources_df = pd.read_csv("data/resources.csv")
learners_df = pd.read_csv("data/learners.csv")
interactions_df = pd.read_csv("data/interactions.csv")

print("Datasets loaded successfully.")
print(f"Concepts: {len(concepts_df)}")
print(f"Prerequisites: {len(prerequisites_df)}")
print(f"Resources: {len(resources_df)}")
print(f"Learners: {len(learners_df)}")
print(f"Interactions: {len(interactions_df)}")

In [ ]:
def create_research_knowledge_graph(concepts, prerequisites, resources, learners, interactions):
    G = nx.DiGraph()

    # 1. Add Concept Nodes
    for _, row in concepts.iterrows():
        node_id = f"Concept:{row['concept_id']}"
        G.add_node(node_id, type='concept', name=row['name'])

    # 2. Add Prerequisite Edges (Concept -> Prerequisite)
    # Definition: Target 'requires' Source (Edge direction: Target -> Prerequisite)
    for _, row in prerequisites.iterrows():
        src = f"Concept:{row['prerequisite_id']}"
        dst = f"Concept:{row['concept_id']}"
        G.add_edge(dst, src, type='requires')

    # 3. Add Resource Nodes & Edges (Resource -> Concept)
    # Definition: Resource 'teaches' Concept
    for _, row in resources.iterrows():
        res_id = f"Resource:{row['resource_id']}"
        con_id = f"Concept:{row['concept_id']}"
        G.add_node(res_id, type='resource', title=row['title'])
        G.add_edge(res_id, con_id, type='teaches')

    # 4. Add Learner Nodes & Goal Edges (Learner -> Concept)
    for _, row in learners.iterrows():
        usr_id = f"Learner:{row['learner_id']}"
        goal_id = f"Concept:{row['target_concept_id']}"
        G.add_node(usr_id, type='learner', name=row['name'])
        # Goal Edge
        if pd.notna(row['target_concept_id']):
             G.add_edge(usr_id, goal_id, type='targets')

    # 5. Add Interaction Edges (Learner -> Resource)
    # These capture the learning history
    for _, row in interactions.iterrows():
        usr_id = f"Learner:{row['learner_id']}"
        res_id = f"Resource:{row['resource_id']}"
        # status ∈ {completed, attempted, failed}
        G.add_edge(usr_id, res_id, type='interacted', status=row['status'], timestamp=row['timestamp'])

    return G

# Build the Graph
kg = create_research_knowledge_graph(concepts_df, prerequisites_df, resources_df, learners_df, interactions_df)

print(f"Knowledge Graph Constructed.")
print(f"Total Nodes: {kg.number_of_nodes()}")
print(f"Total Edges: {kg.number_of_edges()}")

In [ ]:
# --- Explainability Verification ---
# Let's verify the path for the 'Good Learner' vs 'Failing Learner'

# 1. Check Prerequisite Logic
# Does 'Object Oriented Programming' (c4) require 'Functions' (c3)?
# Graph Edge: c4 -> c3 (means c4 depends on c3)
has_prereq = kg.has_edge("Concept:c4", "Concept:c3")
print(f"Explained: OOP (c4) requires Functions (c3)? -> {has_prereq}")

# 2. Analyze 'Failing Learner' (l2)
# Goal: c4. Interaction: Failed r5 (which teaches c4)
learner_id = "Learner:l2"
resource_id = "Resource:r5" # Teaches c4

# Check interaction status
if kg.has_edge(learner_id, resource_id):
    status = kg[learner_id][resource_id]['status']
    print(f"Learner {learner_id} interaction with {resource_id}: {status}")

    # EXPLAINABILITY LOGIC: Check if prerequisites of the target concept were met
    target_concept = "Concept:c4"
    # Edges point Target -> Prerequisite, so successors of Target are its Prerequisites
    prerequisites = list(kg.successors(target_concept)) # c4 -> c3
    print(f"Prerequisites for {target_concept}: {prerequisites}")
    
    # In a real GNN, we'd check if the learner has 'acquired' these prerequisite concepts
    # Here we simply see they are MISSING from the learner's history
    print("Recommendation: The GNN would identify 'Concept:c3' as a missing link causing the failure.")

In [ ]:
import torch
import numpy as np

def prepare_gnn_data_normalized(kg):
    """Prepares data for PyTorch Geometric (Homogeneous for demo)."""
    # Map all string IDs to Integers
    node_to_idx = {node: i for i, node in enumerate(kg.nodes())}
    idx_to_node = {i: node for node, i in node_to_idx.items()}

    edges = list(kg.edges())
    edge_index = [[node_to_idx[edge[0]] for edge in edges],
                  [node_to_idx[edge[1]] for edge in edges]]
    edge_index = torch.tensor(edge_index, dtype=torch.long)

    # Dummy features (Identity)
    num_nodes = len(node_to_idx)
    x = torch.eye(num_nodes) 

    return x, edge_index, node_to_idx

x, edge_index, mapping = prepare_gnn_data_normalized(kg)
print(f"GNN Data Ready: {x.shape} features, {edge_index.shape} edges.")